<a href="https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row = one `content_hash_id`, for one `client_hash_id`, on one `report_date` — a content × day grain in `fact_content_daily_performance`, joined to `dim_content` (static content attributes) on `content_hash_id`.

**Table(s):** `fact_content_daily_performance` (daily metrics: impressions, clicks, sessions, AI-referral counts) joined to `dim_content` (static: content_type, word_count, main_intent, provider_used, etc.) via `content_hash_id`.

**Time window:** February 2025 — a full calendar month, chosen from the confirmed sample date range (2025-01-27 to 2025-02-12+). Full dataset date range unconfirmed due to streaming-scale constraints (78.8M rows); slice selection is based on the confirmed sample range only.

**What I'd predict/rank:** No pre-built label exists in this table (unlike the old CSV's `trend_direction`). The proxy label must be constructed: sum a metric (e.g. `gsc_clicks` or `gsc_impressions`) over a prior sub-window vs. a later sub-window within February, compute % change, then bucket into up/down/stable.

**Deliberately excluded:** All hash IDs (`content_hash_id`, `client_hash_id`, `keyword_hash_id`, `url_hash_id`) — pure identifiers, no predictive signal. Also excluded from the feature set (but used for filtering): `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` — these are availability flags, not content signals.

In [ ]:

!git clone https://github.com/Hadeed07/FlyRank-ML.git
import pandas as pd

fatal: destination path 'FlyRank-ML' already exists and is not an empty directory.


In [ ]:
from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
ds2 = load_dataset("FlyRank/internship-warehouse", "dim_content", streaming=True, split="train")

In [ ]:
sample = list(ds.take(3))
sample[0]

{'report_date': datetime.date(2025, 1, 27),
 'client_hash_id': 'client_9958f0a7ae1df715',
 'content_hash_id': 'content_3b70a18ea133b2bb',
 'client_has_gsc': True,
 'client_has_ga4': True,
 'gsc_data_available': True,
 'ga4_data_available': False,
 'gsc_impressions': 30,
 'gsc_clicks': 0,
 'gsc_sum_position': 115,
 'gsc_avg_position': 3.8333333333333335,
 'ga4_pageviews': 0,
 'ga4_sessions': 0,
 'ga4_users': 0,
 'ga4_engaged_sessions': 0,
 'ga4_total_engagement_sec': 0,
 'sessions_organic': 0,
 'sessions_direct': 0,
 'sessions_referral': 0,
 'sessions_social': 0,
 'sessions_paid': 0,
 'sessions_ai': 0,
 'ai_chatgpt': 0,
 'ai_perplexity': 0,
 'ai_gemini': 0,
 'ai_copilot': 0,
 'ai_claude': 0,
 'ai_meta': 0,
 'ai_other': 0,
 'scroll_events': 0}

In [ ]:
sample = list(ds2.take(3))
sample[0]

{'client_hash_id': 'client_04660893ae39614a',
 'content_hash_id': 'content_004de9653278b5a4',
 'keyword_hash_id': 'keyword_e754999ab88dd9f2',
 'url_hash_id': 'url_d6091f18cf628794',
 'keyword_char_count': 22,
 'keyword_token_count': 4,
 'url_char_count': 108,
 'content_created_date': datetime.date(2026, 5, 30),
 'content_updated_date': datetime.date(2026, 7, 1),
 'content_type': 'keyword article',
 'search_volume': 30,
 'competition': 0.91,
 'competition_level': 'HIGH',
 'cpc': 0.98,
 'main_intent': 'transactional',
 'backlinks': 16,
 'category_count': 3,
 'keyword_created_date': datetime.date(2026, 5, 12),
 'provider_used': 'gemini-generate-content',
 'model_used': 'gemini-3-flash-preview',
 'char_count': 15682,
 'word_count': 2555,
 'last_optimized_date': None,
 'optimization_eligible_date': None,
 'is_published': True,
 'is_deleted': False}

In [ ]:
import itertools
rows = list(itertools.islice(ds, 5000))
dates = [r['report_date'] for r in rows]
print(min(dates), max(dates))

In [ ]:
def in_feb_2025(row):
    return row['report_date'].year == 2025 and row['report_date'].month == 2

feb_ds = ds.filter(in_feb_2025)
feb_rows = list(itertools.islice(feb_ds, 20_000))  # cap so it doesn't run forever
import pandas as pd
feb_df = pd.DataFrame(feb_rows)
feb_df.shape

Checking the duplicates: Grain Check

In [ ]:
print("Total rows:", len(feb_df))
print("Unique (content_hash_id, report_date) pairs:", feb_df.drop_duplicates(['content_hash_id','report_date']).shape[0])

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


**Features — from `fact_content_daily_performance`, prior window (Sept 1–15) only, aggregated per `content_hash_id`**
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`
- `gsc_days_with_data` (derived: `gsc_sum_position / gsc_avg_position`, rounded — recovers day-count so the raw sum becomes interpretable)
- `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`
- `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`, `sessions_ai`
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`
- `scroll_events`

**Features — from `dim_content` (static + point-in-time as of Sept 15)**
- `content_type`, `main_intent`, `provider_used`, `model_used`
- `search_volume`, `competition`, `cpc`, `backlinks`, `category_count`
- `word_count`, `keyword_char_count`, `keyword_token_count`, `url_char_count`
- `is_published`, `is_deleted`
- `content_age_days` (derived: Sept 15 − `content_created_date` — safe, set-once field)
- `keyword_age_days` (derived: Sept 15 − `keyword_created_date` — safe, set-once field)
- `is_optimization_eligible_yet` (derived: `optimization_eligible_date` ≤ Sept 15)

**Context (used for filtering/grouping, never as a feature)**
- `client_hash_id` — grouping key for the train/test split
- `report_date` — defines the prior/later windows, not itself a feature
- `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` — flags for which rows are trustworthy

**Excluded**
- `content_hash_id`, `keyword_hash_id`, `url_hash_id` — pure identifiers, no predictive signal
- `gsc_impressions`, later window — this is the label source (two-gate rule, gate 1)
- `competition_level` — redundant tier derived from `competition`
- `char_count` — redundant with `word_count`
- `gsc_sum_position` — superseded by derived `gsc_days_with_data`
- `days_since_last_optimized` — 0% coverage after leakage-safe gating (14,812/53,127
  rows would leak by being optimized after the cutoff; the remaining 38,315 were never
  optimized at all as of this slice); not usable as a feature.
- `is_optimization_eligible_yet` — 100% of rows resolve to `False` as of the Sept 15
  cutoff (0/53,127 eligible on or before cutoff); constant value, zero variance, no
  predictive signal regardless of coverage. Also noted: identical counts to
  `last_optimized_date`'s gating check, suggesting the two columns may share an
  underlying source event.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Result — `content_updated_date` gating check:**
52,855 / 53,127 rows (99.5%) would leak future information under a naive (ungated)
`days_since_update` calculation — i.e. the update happened after the Sept 15 cutoff.
After gating, only 272 rows (0.5%) retain a valid value.

**Result — `last_optimized_date` gating check:**
Of 53,127 rows: 14,812 (27.9%) would leak (optimized after the cutoff), and 38,315
(72.1%) have no optimization date at all (never optimized as of this slice). Together
these account for 100% of rows — after gating, **zero rows** have a valid
`days_since_last_optimized` value.

`is_optimization_eligible_yet` gating check:**
0 / 53,127 rows are eligible on or before Sept 15. The 14,812 rows eligible after the
cutoff resolve to `False` (not yet eligible), and the 38,315 rows with no eligibility
date (NaT) also resolve to `False` (no eligibility set). Combined: **100% of rows are
`False` as of this cutoff** — zero variance, not missingness

**Conclusion:** Both derived date features are dropped. The gating logic itself is
proven correct — it successfully caught and nulled out every leaking value in both
cases — but the underlying columns don't support a usable feature at this cutoff:
nearly all content in this slice was either updated or optimized *after* September 15,
not before it. This suggests content updates/optimization in this dataset cluster
later in a content's lifecycle relative to any early-to-mid window, which is itself
worth noting as a data-limits observation in Section 4 rather than just a dropped
feature.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


- **No calendar timestamps.** Only day-count fields exist (`content_age_days`,
  `days_since_last_update`, `days_with_impressions`, `days_with_sessions`). It's
  impossible to confirm whether all rows share a single snapshot date or were
  captured at different points in time. A true calendar-based time-aware split
  is therefore not possible; the client-grouped split is the primary leakage
  safeguard available.

- **Unconfirmed source windows for several fields.** `ctr`, `avg_position`,
  `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `days_with_impressions`,
  and `days_with_sessions` carry no window suffix, so their exact source period
  (prev_30d vs. last_30d vs. full 90d) is unconfirmed. They are held out of the
  feature set until verified.

- **`new`/`flat` are excluded on structural reasoning, not confirmed data.**
  These two `trend_direction` classes are treated as zero-history artifacts
  based on how they were reverse-engineered from the data, but the missing-values
  scan that would formally confirm this hasn't been run in this session.

- **This is a pre-aggregated snapshot, not raw daily data.** The dataset has
  already been rolled up per content_id; nothing here supports day-by-day or
  event-level analysis, only window-level comparisons.

- **Observational, not causal.** Nothing in this data supports claims like
  "a refresh caused a recovery," or attributing movement to specific actions
  (algorithm changes, AI citations, etc.). Any output is directional and
  decision-support only, meant to prioritize human review, not to explain
  *why* a page is trending a certain way.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.